In [4]:
from hf_auth import init_hf
init_hf()

import numpy as np
import ollama
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. EXTRACT & CHUNK PIPELINE

def extract_text(pdf_path: str) -> str:
    """Extract raw text from a PDF."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


def fixed_size_chunking(
    text: str, chunk_size: int = 500, overlap: int = 50
) -> list[str]:
    """Split text into fixed-size chunks with character overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


# Load and process the PDF
manual_path = "op_in_2_router.pdf"
raw_text = extract_text(manual_path)
chunks = fixed_size_chunking(raw_text, chunk_size=500, overlap=50)

# Print chunks to inspect sentence/table splits
print(f"--- Total Chunks: {len(chunks)} ---")
for i, chunk in enumerate(chunks[:3]):  # Preview first 3
    print(f"\n[Chunk {i}]:\n{chunk}\n{'-'*30}")


# RETRIEVAL METHODS SETUP

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(chunks)


def retrieve_tfidf(query: str) -> tuple[int, str]:
    query_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    best_idx = int(np.argmax(scores))
    return best_idx, chunks[best_idx]


# Dense Embeddings
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embed_model.encode(chunks)


def retrieve_embedding(query: str) -> tuple[int, str]:
    query_vec = embed_model.encode([query])
    scores = cosine_similarity(query_vec, chunk_embeddings).flatten()
    best_idx = int(np.argmax(scores))
    return best_idx, chunks[best_idx]


# TEST QUESTIONS & COMPARISON

questions = [
    "How do I connect the router to WiFi?",
    "My internet box won't connect to my network",
    "How do I reset it?",
    "What's your return policy?",
    "What temperature range does the thermostat support?", 
]

print("\n" + "=" * 50)
print("RETRIEVAL COMPARISON")
print("=" * 50)

for q in questions:
    tfidf_idx, _ = retrieve_tfidf(q)
    emb_idx, _ = retrieve_embedding(q)
    print(f"\nQuestion: '{q}'")
    print(f"  -> TF-IDF matched:    Chunk {tfidf_idx}")
    print(f"  -> Embedding matched: Chunk {emb_idx}")
    print(
        f"  -> Match status:      {'SAME' if tfidf_idx == emb_idx else 'DIFFERENT'}"
    )


# GENERATION WITH LOCAL LLM

# Test the reformulated query with the chunk retrieved by embeddings
best_query = "Where is the reset button"
best_chunk_id, best_chunk = retrieve_embedding(best_query)

prompt = f"""Use ONLY the following context to answer the question. If the answer cannot be found in the context, say so.

Context:
{best_chunk}

Question: {best_query}
Answer:"""

print("\n" + "=" * 50)
print("LOCAL LLM RESPONSE")
print("=" * 50)

response = ollama.generate(model="llama3.2", prompt=prompt)
print(response["response"])

✅ Hugging Face token successfully loaded!
--- Total Chunks: 109 ---

[Chunk 0]:
November 2007
208-10148-01 
v1.0
NETGEAR, Inc.
4500 Great America Parkway 
Santa Clara, CA 95054 USA
Wireless Router Setup 
Manual

ii
Trademarks
NETGEAR and the NETGEAR logo are registered trademarks, and RangeMax and Smart Wizard are trademarks of 
NETGEAR. Inc. Microsoft, Windows, and Windows NT are registered trademarks of Microsoft Corporation. Other 
brand and product names are registered trademarks or trademarks of their respective holders.
Statement of Conditions
In the interest of impro
------------------------------

[Chunk 1]:
.
Statement of Conditions
In the interest of improving internal design, operational function, and/or reliability, NETGEAR reserves the right to 
make changes to the products described in this document without notice.
NETGEAR does not assume any liability that may occur due to the use or application of the product(s) or circuit 
layout(s) described herein.
© 2007 by NETGEAR,

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


RETRIEVAL COMPARISON

Question: 'How do I connect the router to WiFi?'
  -> TF-IDF matched:    Chunk 31
  -> Embedding matched: Chunk 38
  -> Match status:      DIFFERENT

Question: 'My internet box won't connect to my network'
  -> TF-IDF matched:    Chunk 31
  -> Embedding matched: Chunk 63
  -> Match status:      DIFFERENT

Question: 'How do I reset it?'
  -> TF-IDF matched:    Chunk 44
  -> Embedding matched: Chunk 100
  -> Match status:      DIFFERENT

Question: 'What's your return policy?'
  -> TF-IDF matched:    Chunk 9
  -> Embedding matched: Chunk 101
  -> Match status:      DIFFERENT

Question: 'What temperature range does the thermostat support?'
  -> TF-IDF matched:    Chunk 104
  -> Embedding matched: Chunk 97
  -> Match status:      DIFFERENT

LOCAL LLM RESPONSE
The reset button is located on the rear panel of the router, specifically on the "Restore factory settings" button.
